# Echo: Global Media Sentiment Monitor

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, beginner familiarity with NLP concepts, and comfort with communications metrics such as sentiment, prominence, and outlet tiers.

            **What You Will Learn:**

            - Describe how article tone, prominence, and outlet mix influence communications escalation decisions.
- Compare a simple newsroom triage rule against a text-classification pipeline for article-priority scoring.
- Adapt a media-monitoring workflow into a strategic briefing pattern for another stakeholder group.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking notebooks that translate raw signals into actionable reports for deal teams, communications, risk, legal, and integrity functions.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** Large organizations can receive tens of thousands of mentions, and communications teams need to surface the right stories before the narrative hardens.

**Operational question:** Which articles should the communications team elevate immediately, and what kind of coverage is driving the urgency?

**Primary stakeholders:** communications teams, investor-relations leads, corporate affairs groups, and senior leadership

**Decision supported:** Turn raw coverage into a ranked queue with outlet context, prominence, and a concise briefing lens.

**Comprehension check:** Before looking at the data, write one sentence describing why a high-prominence neutral article might still deserve escalation.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use synthetic article text plus lightweight NLP components so the workflow remains portable and easy to inspect.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

print("Environment ready for a synthetic media-monitoring workflow.")


## Step 3 — Data Creation & Context

We simulate article headlines, outlet types, prominence levels, and organization focus. The text is synthetic but preserves the communication trade-offs that shape priority queues.


In [ ]:
outlet_types = ["top_tier", "wire", "trade_press", "regional"]
tone_banks = {
    "positive": ["beats guidance", "wins mandate", "expands partnership", "upgrades outlook"],
    "neutral": ["hosts investor day", "announces update", "publishes filing", "comments on market"],
    "negative": ["faces probe", "cuts forecast", "misses target", "draws criticism"],
}
focus_banks = ["organization focused", "sector mention", "peer comparison", "brief reference"]
records = []

for _ in range(1600):
    tone = RNG.choice(["positive", "neutral", "negative"], p=[0.28, 0.44, 0.28])
    outlet = RNG.choice(outlet_types, p=[0.24, 0.18, 0.28, 0.30])
    prominence = RNG.uniform(0.1, 1.0)
    focus = RNG.choice(focus_banks, p=[0.45, 0.20, 0.18, 0.17])
    headline = f"company {RNG.choice(tone_banks[tone])} in {outlet} coverage with {focus}"
    urgency = 0.9 * (tone == "negative") + 0.6 * prominence + 0.4 * (outlet in {"top_tier", "wire"}) + 0.3 * (focus == "organization focused") + RNG.normal(0, 0.15)
    if urgency >= 1.35:
        priority = "High"
    elif urgency >= 0.8:
        priority = "Medium"
    else:
        priority = "Low"
    records.append((headline, outlet, prominence, focus, priority))

article_df = pd.DataFrame(
    records,
    columns=["headline", "outlet_type", "prominence_score", "focus_type", "priority_bucket"],
)
article_df["article_text"] = (
    article_df["headline"]
    + " outlet_" + article_df["outlet_type"]
    + " focus_" + article_df["focus_type"].str.replace(" ", "_")
)

print(article_df.head(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Inspect the priority mix and where those articles come from before training a classifier. This helps connect media strategy to the later NLP workflow.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=article_df, x="priority_bucket", order=["Low", "Medium", "High"], color="#2A9D8F", ax=axes[0])
axes[0].set_title("Media Monitor: Priority Bucket Distribution")
axes[0].set_xlabel("Priority Bucket")
axes[0].set_ylabel("Article Count")

outlet_priority = (
    article_df.groupby(["outlet_type", "priority_bucket"])
    .size()
    .reset_index(name="count")
)
sns.barplot(
    data=outlet_priority,
    x="outlet_type",
    y="count",
    hue="priority_bucket",
    hue_order=["Low", "Medium", "High"],
    ax=axes[1],
)
axes[1].set_title("Priority Mix by Outlet Type")
axes[1].set_xlabel("Outlet Type")
axes[1].set_ylabel("Article Count")

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The charts show how article priority spreads across outlet types, emphasizing that top-tier and wire coverage can be strategically important even when article counts are lower.


## Step 5 — Feature Engineering

We keep the feature engineering visible by combining raw text with outlet and focus tokens instead of hiding context inside an opaque pipeline.


In [ ]:
analysis_df = article_df.copy()
analysis_df["prominence_band"] = pd.cut(
    analysis_df["prominence_score"],
    bins=[0.0, 0.35, 0.7, 1.0],
    labels=["low_prominence", "mid_prominence", "high_prominence"],
    include_lowest=True,
)
analysis_df["model_text"] = (
    analysis_df["article_text"]
    + " "
    + analysis_df["prominence_band"].astype(str)
)

print(analysis_df[["headline", "outlet_type", "prominence_band", "priority_bucket"]].head(5).to_string(index=False))


## Step 6 — Baseline Establishment

A common baseline is a newsroom rule: escalate high-prominence pieces from major outlets, and everything else waits for manual review.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df["model_text"],
    analysis_df["priority_bucket"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["priority_bucket"],
)

test_meta = analysis_df.loc[X_test.index, ["prominence_score", "outlet_type"]]
baseline_pred = np.where(
    (test_meta["prominence_score"] >= 0.72)
    & (test_meta["outlet_type"].isin(["top_tier", "wire"])),
    "High",
    np.where(test_meta["prominence_score"] >= 0.45, "Medium", "Low"),
)

print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")


## Step 7 — Model Training & Evaluation

Train a text model that reads headline phrasing plus outlet context. Prediction prompt: which words or outlet cues do you expect to push the model toward `High` priority?


In [ ]:
priority_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3)),
        ("classifier", LogisticRegression(max_iter=1200)),
    ]
)
priority_model.fit(X_train, y_train)
model_pred = priority_model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_pred):.3f}")
print(classification_report(y_test, model_pred))


## Step 8 — Interpretability & Explainability

Top-weighted terms help the communications team check whether the model is elevating the same article patterns that a senior press officer would care about.


In [ ]:
vectorizer = priority_model.named_steps["tfidf"]
classifier = priority_model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()

for class_index, class_name in enumerate(classifier.classes_):
    top_terms = feature_names[np.argsort(classifier.coef_[class_index])[-8:]][::-1]
    print(f"Top signals for {class_name}: {', '.join(top_terms)}")


## Step 9 — Operational Application

Use the classifier to assemble a briefing queue with priority counts, outlet mix, and the top headlines a communications lead should read first.


In [ ]:
latest_articles = analysis_df.sample(12, random_state=RANDOM_SEED).copy()
latest_articles["predicted_priority"] = priority_model.predict(latest_articles["model_text"])

queue_summary = (
    latest_articles.groupby(["predicted_priority", "outlet_type"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["predicted_priority", "article_count"], ascending=[True, False])
)

high_priority = latest_articles.loc[
    latest_articles["predicted_priority"] == "High",
    ["headline", "outlet_type", "prominence_score"],
].sort_values("prominence_score", ascending=False)

print("Briefing summary:")
print(queue_summary.to_string(index=False))
print("\nHeadlines to review first:")
print(high_priority.head(5).round(3).to_string(index=False))


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic media-monitoring workflow that turns raw article text, outlet context, and prominence into a ranked communications queue.

            **Limitations to note:**

            - The text is synthetic and far simpler than multilingual, real-world media coverage.
- Prominence is simulated with a single score rather than a richer page-position or broadcast-reach measure.
- The workflow supports communications triage, not legal, reputational, or investment decisions on its own.

            **Ethical reflection:** A media-monitoring model can overreact to noisy criticism or underweight local coverage, so organizations should avoid using it as an automated truth detector.

            **Reflection questions:**

            1. How would you extend the pipeline to handle multiple languages without collapsing important local nuance?
2. When should a high-prominence neutral article still move to the top of the queue?
3. What additional metadata would improve a board-level communications briefing?


            ## References

            - Jurafsky, D., & Martin, J. H. Speech and Language Processing.
- Scikit-learn user guide: text feature extraction and linear models.
- Scenario inspiration: media-intelligence workflows used in communications and investor-relations monitoring.
